# YouTube Trending Videos - Exploratory Data Analysis

**Author:** Meeta
**Dataset:** [YouTube Videos Dataset - Kaggle](https://www.kaggle.com/datasets/rajatrc1705/youtube-videos-dataset)
**Tools:** Python, Pandas, Matplotlib, Seaborn

---

## Project Overview

This notebook performs a structured exploratory data analysis (EDA) on a dataset of ~3,400 YouTube trending videos.

**Key questions explored:**
1. Which content categories dominate YouTube trending?
2. How clean is the data - missing values, duplicates?
3. What patterns exist in video titles?
4. What actionable insights can we draw?

---

## 1. Setup & Data Loading

We load the dataset directly from Kaggle using `kagglehub`, then read it into a Pandas DataFrame for analysis.

In [ ]:
# --- Import all required libraries ---
# kagglehub: to download dataset directly from Kaggle
# pandas: for data manipulation and analysis
# matplotlib & seaborn: for visualisation
# warnings: to suppress non-critical warnings for cleaner output

import kagglehub
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set a clean visual style for all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120  # Higher resolution plots

# Download the YouTube videos dataset from Kaggle
path = kagglehub.dataset_download('rajatrc1705/youtube-videos-dataset')
print(f'Dataset downloaded to: {path}')

In [ ]:
# Load the CSV file into a Pandas DataFrame
# We use the dynamic 'path' variable so the notebook works on any machine
df = pd.read_csv(f'{path}/youtube.csv')

# Quick sanity check - how large is the dataset?
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Columns: {list(df.columns)}')

---
## 2. Initial Data Inspection

Before jumping into analysis, we always take a first look at the raw data.
This helps us understand the structure, spot any obvious issues, and get a feel for the content.

In [ ]:
# Display the first 5 rows to understand the structure
# We can see column names, data types at a glance, and sample values
df.head()

In [ ]:
# df.info() gives us:
# - Column names and their data types
# - Number of non-null values per column (useful for spotting missing data)
# - Memory usage of the DataFrame
df.info()

In [ ]:
# Inspect a real example of the text fields
# This helps us understand what 'title' and 'description' actually look like
# before we run any analysis on them
print('--- Sample Title ---')
print(df['title'].iloc[0])
print()
print('--- Sample Description (first 300 chars) ---')
print(str(df['description'].iloc[0])[:300])

---
## 3. Data Quality Assessment

Clean data = reliable insights. Before analysing anything, we systematically check for:
- **Missing values** - nulls can skew results or break operations
- **Duplicate rows** - duplicates inflate counts and distort distributions

We fix any issues found before moving on.

In [ ]:
# Count missing values per column
# We also calculate the percentage so we know how serious each gap is
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

# Combine into a summary table and filter to only show columns with missing data
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('No missing values found - data is complete!')
else:
    print('Columns with missing values:')
    display(missing_df)

In [ ]:
# Check for fully duplicate rows (every column identical)
n_dupes = df.duplicated().sum()
print(f'Duplicate rows found: {n_dupes}')

# Remove duplicates and reset the index so it stays sequential (0, 1, 2...)
# drop=True prevents the old index from becoming a column
df = df.drop_duplicates().reset_index(drop=True)
print(f'Dataset size after deduplication: {len(df):,} rows')

---
## 4. Category Distribution

Which video categories appear most frequently in YouTube trending?
Understanding this tells us what type of content the YouTube algorithm tends to surface - useful for creators and analysts alike.

In [ ]:
# Count how many videos belong to each category
# value_counts() automatically sorts from most to least frequent
category_counts = df['category'].value_counts()

print('Category distribution:')
print(category_counts.to_string())

In [ ]:
# Visualise as a horizontal bar chart
# Horizontal bars work better than vertical when category names are long
fig, ax = plt.subplots(figsize=(10, 6))

# Use a muted colour palette - one colour per category
colors = sns.color_palette('muted', len(category_counts))

# Sort ascending so the longest bar is at the top (more readable)
category_counts.sort_values().plot(kind='barh', ax=ax, color=colors, edgecolor='white')

ax.set_title('Number of Trending Videos by Category', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Video Count', fontsize=12)
ax.set_ylabel('Category', fontsize=12)

# Annotate each bar with its exact count for precision
for bar in ax.patches:
    ax.text(
        bar.get_width() + 5,                    # position: just right of the bar end
        bar.get_y() + bar.get_height() / 2,     # vertically centred on the bar
        f'{int(bar.get_width()):,}',             # formatted number with comma separator
        va='center', fontsize=9
    )

plt.tight_layout()
plt.show()

---
## 5. Title Length Analysis

Video titles are one of the most important factors for click-through rate.
We examine how long (in characters) trending video titles tend to be - this can reveal patterns in how successful creators write their titles.

In [ ]:
# Create a new column with the character length of each title
# We cast to str first to handle any unexpected non-string values safely
df['title_length'] = df['title'].astype(str).apply(len)

# describe() gives us count, mean, std, min, quartiles, and max in one go
print('Title length stats (characters):')
print(df['title_length'].describe().round(1))

In [ ]:
# Plot the distribution of title lengths as a histogram
# This shows us the most common title length range at a glance
fig, ax = plt.subplots(figsize=(10, 4))

# 40 bins gives enough detail without being too noisy
ax.hist(df['title_length'], bins=40, color='steelblue', edgecolor='white', alpha=0.85)

# Overlay median and mean lines so we can compare them
# If mean > median, the distribution is right-skewed (a few very long titles pulling it up)
ax.axvline(df['title_length'].median(), color='crimson', linestyle='--', linewidth=1.5,
           label=f"Median: {df['title_length'].median():.0f} chars")
ax.axvline(df['title_length'].mean(), color='darkorange', linestyle='--', linewidth=1.5,
           label=f"Mean: {df['title_length'].mean():.1f} chars")

ax.set_title('Distribution of Video Title Lengths', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Title Length (characters)', fontsize=12)
ax.set_ylabel('Number of Videos', fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

---
## 6. Key Insights & Next Steps

| # | Insight |
|---|---|
| 1 | **Category dominance** - a small number of categories account for most trending videos, suggesting the algorithm heavily favours certain content types. |
| 2 | **Data quality was good** - minimal missing values; a small number of duplicate rows were identified and removed before analysis. |
| 3 | **Title length** - most trending titles fall between 30-70 characters, suggesting creators optimise for clarity without being too wordy. |

### Next Steps
Given more time, I would explore:
- **Engagement metrics** - views, likes, comment counts across categories
- **NLP on titles/descriptions** - keyword frequency, sentiment analysis
- **Temporal trends** - what time of day or day of week do videos tend to trend?
- **Predictive modelling** - can we predict trending potential from video metadata?

---
*This notebook is part of Meeta's data analysis portfolio.*